In [1]:
import pandas as pd

In [ ]:
"""
Объединение синтетических survey-датасетов (d002, d006, d008, kv_vopr_*)

Ключи:
    NOMER               = год (4 цифры) + id домохозяйства (6 цифр), стабилен по годам
    NOMER + NOMP         = ключ человека внутри домохозяйства

Скрипт:
  1. читает все файлы;
  2. добавляет колонки YEAR / HOUSEHOLD_ID, вычисленные из NOMER;
  3. строит уровень домохозяйства (household_wide) — d006 + агрегаты по d002/d008/kv_vopr;
  4. строит уровень человека (person_wide) — d008 + d002_subject + d002_ocenka;
  5. агрегирует все kv_vopr_* (расходы) в одну таблицу по NOMER+KVARTAL+GOD;
  6. сохраняет результаты в CSV.

Меняйте только блок CONFIG под свои пути / нужный набор таблиц.
"""

import pandas as pd
from pathlib import Path

# ------------------------- CONFIG -------------------------
DATA_DIR = Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004")
OUT_DIR = Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/D00X")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "d008": "combined_data_d008.csv",              # ростер (человек)
    "d006": "combined_data_d006.csv",               # жильё (домохозяйство)
    "d002_subject": "combined_data_d002_subject.csv",  # анкета (человек)
    "d002_ocenka": "combined_data_d002_ocenka.csv",    # оценки (человек)
}
# все таблицы kv_vopr_0..12 — расходы/вопросы бюджета, уровень домохозяйства (по кварталам)
KV_VOPR_FILES = sorted(DATA_DIR.glob("kv_vopr_*_combined.csv"))


def add_year_household(df: pd.DataFrame, nomer_col: str = "NOMER") -> pd.DataFrame:
    """Достаём YEAR и HOUSEHOLD_ID из NOMER (первые 4 цифры = год, остальное = id семьи)."""
    df = df.copy()
    nomer_str = df[nomer_col].astype(str)
    df["YEAR"] = nomer_str.str[:4].astype(int)
    df["HOUSEHOLD_ID"] = nomer_str.str[4:]
    return df


def load(name: str) -> pd.DataFrame:
    #path = DATA_DIR / FILES[name]
    df = pd.read_csv(name, low_memory=False)
    return add_year_household(df)


# ------------------------- 1. ЗАГРУЗКА -------------------------
d008 = load("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d008/combined_data_d008.csv")
d006 = load("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d006/combined_data_d006.csv")
d002_subject = load("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d002/combined_data_d002_subject.csv")
d002_ocenka = load("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d002/combined_data_d002_ocenka.csv")

print(f"d008: {d008.shape}, d006: {d006.shape}, "
      f"d002_subject: {d002_subject.shape}, d002_ocenka: {d002_ocenka.shape}")

# ------------------------- 2. УРОВЕНЬ ЧЕЛОВЕКА -------------------------
# join по NOMER + NOMP + YEAR (год нужен явно, т.к. NOMP переиспользуется в разных годах)
person_key = ["NOMER", "NOMP", "YEAR", "HOUSEHOLD_ID"]

person_wide = d008.merge(
    d002_subject.drop(columns=["TE", "K"], errors="ignore"),
    on=person_key, how="left", suffixes=("", "_subj")
).merge(
    d002_ocenka.drop(columns=["TE", "K"], errors="ignore"),
    on=person_key, how="left", suffixes=("", "_ocen")
)

print(f"person_wide: {person_wide.shape}")

# ------------------------- 3. РАСХОДЫ (kv_vopr_*) -------------------------
# Файлы большие (kv_vopr_1 сам по себе 4.4 млн строк, всего ~8.6 млн),
# поэтому НЕ держим их все в памяти разом: читаем по одному, сразу
# агрегируем сумму трат (STOIMK) по домохозяйство+год+квартал и суммируем
# результат по всем таблицам. Полный long-файл пишем сразу на диск (append),
# а не собираем в памяти.
long_out_path = OUT_DIR / "kv_vopr_all_long.csv"
first_write = True
agg_parts = []

for path in KV_VOPR_FILES:
    df = pd.read_csv(path, low_memory=False)
    df = add_year_household(df)
    df["SOURCE_TABLE"] = path.stem  # напр. kv_vopr_3_combined

    # пишем построчный (long) файл инкрементально, без хранения всего в RAM
    df.to_csv(long_out_path, mode="w" if first_write else "a",
              header=first_write, index=False)
    first_write = False

    if "STOIMK" in df.columns:
        part = (
            df.groupby(["HOUSEHOLD_ID", "YEAR", "KVARTAL"], dropna=False)["STOIMK"]
            .sum(min_count=1)
            .reset_index()
        )
        agg_parts.append(part)

    print(f"  обработан {path.name}: {df.shape}")
    del df

# суммируем агрегаты по всем таблицам (в каждой таблице свой набор трат)
expenditure_by_hh_q = (
    pd.concat(agg_parts, ignore_index=True)
    .groupby(["HOUSEHOLD_ID", "YEAR", "KVARTAL"], dropna=False)["STOIMK"]
    .sum(min_count=1)
    .reset_index()
    .rename(columns={"STOIMK": "TOTAL_EXPENDITURE"})
)

print(f"expenditure_by_hh_q: {expenditure_by_hh_q.shape}")

# ------------------------- 4. УРОВЕНЬ ДОМОХОЗЯЙСТВА -------------------------
household_wide = d006.merge(
    expenditure_by_hh_q,
    on=["HOUSEHOLD_ID", "YEAR"], how="left"
)

print(f"household_wide: {household_wide.shape}")

# ------------------------- 5. СОХРАНЕНИЕ -------------------------
# kv_vopr_all_long.csv уже записан инкрементально в блоке 3
person_wide.to_csv(OUT_DIR / "person_wide.csv", index=False)
household_wide.to_csv(OUT_DIR / "household_wide.csv", index=False)
expenditure_by_hh_q.to_csv(OUT_DIR / "expenditure_by_household_quarter.csv", index=False)

print("Готово. Файлы сохранены в", OUT_DIR)


d008: (169059, 37), d006: (46743, 249), d002_subject: (47771, 115), d002_ocenka: (47771, 123)
person_wide: (169059, 263)
  обработан kv_vopr_0_combined.csv: (192000, 9)
  обработан kv_vopr_10_combined.csv: (166793, 15)
  обработан kv_vopr_11_combined.csv: (396510, 36)
  обработан kv_vopr_12_combined.csv: (91166, 15)
  обработан kv_vopr_1_combined.csv: (4391820, 13)
  обработан kv_vopr_2_combined.csv: (998776, 12)
  обработан kv_vopr_3_combined.csv: (422130, 11)
  обработан kv_vopr_4_combined.csv: (37119, 11)
  обработан kv_vopr_5_combined.csv: (74202, 11)
  обработан kv_vopr_6_combined.csv: (675523, 13)
  обработан kv_vopr_7_combined.csv: (248753, 11)
  обработан kv_vopr_9_combined.csv: (917692, 23)
expenditure_by_hh_q: (189106, 4)
household_wide: (184474, 251)
Готово. Файлы сохранены в /content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/D00X


In [2]:
"""
Объединение ВСЕХ датасетов (d002, d004=kv_vopr_*, d006, d008)
с сохранением результата отдельными файлами по годам.

ВАЖНО про колонки-омонимы (GR1, GR2, ..., и любые другие одинаковые
имена в разных датасетах): они переименовываются с префиксом источника,
например GR1 -> d004_vopr9__GR1 / d004_vopr11__GR1 / d002_subject__GR1,
т.к. одинаковое имя в разных таблицах означает РАЗНЫЕ вещи.
Ключевые колонки (NOMER, NOMP, GOD/GODO, KVARTAL, YEAR, HOUSEHOLD_ID)
НЕ переименовываются — по ним и идёт join.

Результат на каждый год:
    master_wide_{year}.csv     — человек x квартал, с d008+d002+d006
                                  и суммой расходов (TOTAL_EXPENDITURE)
    kv_vopr_detail_{year}.csv  — все 12 таблиц kv_vopr за этот год,
                                  построчно (concat), с префиксами колонок
                                  и меткой SOURCE_TABLE — тут сохранена
                                  полная детализация (в master_wide только
                                  агрегированная сумма трат)
"""

import pandas as pd
from pathlib import Path

# ------------------------- CONFIG -------------------------
# Каждый датасет лежит в своей папке — указываем ПОЛНЫЙ путь до каждого
# файла напрямую, а не общий DATA_DIR + имя файла.
OUT_DIR = Path("/content")
OUT_DIR.mkdir(parents=True, exist_ok=True)


FILES = {
    "d008": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d008/combined_data_d008.csv"),
    "d006": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d006/combined_data_d006.csv"),
    "d002_subject": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d002/combined_data_d002_subject.csv"),
    "d002_ocenka": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d002/combined_data_d002_ocenka.csv"),
}

# kv_vopr (= форма d004): свой путь на каждую таблицу. Ключ словаря — номер
# вопроса (используется как d004_vopr{n}__ при переименовании колонок).
KV_VOPR_FILES = {
    "0": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004/kv_vopr_0_combined.csv"),
    "1": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004/kv_vopr_1_combined.csv"),
    "2": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004/kv_vopr_2_combined.csv"),
    "3": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004/kv_vopr_3_combined.csv"),
    "4": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004/kv_vopr_4_combined.csv"),
    "5": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004/kv_vopr_5_combined.csv"),
    "6": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004/kv_vopr_6_combined.csv"),
    "7": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004/kv_vopr_7_combined.csv"),
    "9": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004/kv_vopr_9_combined.csv"),
    "10": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004/kv_vopr_10_combined.csv"),
    "11": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004/kv_vopr_11_combined.csv"),
    "12": Path("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/d004/kv_vopr_12_combined.csv"),
}

# колонки-ключи, которые никогда не переименовываем (по ним джойним).
# TE, K, NOMER — общие для ВСЕХ датасетов и несут одну и ту же информацию,
# поэтому тоже не префиксуются (и не дублируются при джойне — оставляем
# только одну копию, см. ниже).
KEY_COLS = {"NOMER", "NOMP", "GOD", "GODO", "KVARTAL", "YEAR", "HOUSEHOLD_ID", "TE", "K"}


def add_year_household(df: pd.DataFrame, nomer_col: str = "NOMER") -> pd.DataFrame:
    """NOMER = год (4 цифры) + id домохозяйства (остальное) -> 2 колонки."""
    df = df.copy()
    nomer_str = df[nomer_col].astype(str)
    df["YEAR"] = nomer_str.str[:4].astype(int)
    df["HOUSEHOLD_ID"] = nomer_str.str[4:]
    return df


def prefix_columns(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """Переименовывает все НЕ-ключевые колонки в {prefix}__{col},
    чтобы одинаково названные, но разные по смыслу поля (напр. GR1)
    из разных таблиц не путались при объединении."""
    rename_map = {c: f"{prefix}__{c}" for c in df.columns if c not in KEY_COLS}
    return df.rename(columns=rename_map)


def load(name: str) -> pd.DataFrame:
    df = add_year_household(pd.read_csv(FILES[name], low_memory=False))
    return prefix_columns(df, name)


# ------------------------- 1. d008 + d002 -> УРОВЕНЬ ЧЕЛОВЕКА -------------------------
d008 = load("d008")
d002_subject = load("d002_subject")
d002_ocenka = load("d002_ocenka")

person_key = ["NOMER", "NOMP", "YEAR", "HOUSEHOLD_ID"]

# TE, K одинаковые во всех датасетах — оставляем только одну копию (из d008),
# чтобы merge не плодил TE_x/TE_y из-за одинаковых незапрефиксованных имён.
person_wide = d008.merge(
    d002_subject.drop(columns=["TE", "K"], errors="ignore"), on=person_key, how="left"
).merge(
    d002_ocenka.drop(columns=["TE", "K"], errors="ignore"), on=person_key, how="left"
)
print(f"person_wide (d008+d002): {person_wide.shape}")

# ------------------------- 2. d006 -> УРОВЕНЬ ДОМОХОЗЯЙСТВА -------------------------
d006 = load("d006")
print(f"d006: {d006.shape}")

# ------------------------- 3. kv_vopr_* (=d004) -> по годам -------------------------
# Каждая таблица kv_vopr_N получает СВОЙ префикс (d004_vopr0, d004_vopr1, ...),
# т.к. GR1..GR9 в kv_vopr_9 и GR1..GR23 в kv_vopr_11 — это разные признаки,
# даже несмотря на одинаковые имена.
# Файлы большие (kv_vopr_1 = 4.4 млн строк), поэтому НЕ копим все данные в
# памяти по годам — в одном проходе по файлу сразу дописываем нужный год
# в соответствующий kv_vopr_detail_{year}.csv (mode="a").
MONEY_COLS = ["STOIMK", "PRICE"]  # в разных таблицах трата названа по-разному

detail_files_started = set()   # year -> уже создан ли kv_vopr_detail_{year}.csv
agg_by_year = {}                # year -> список агрегатов расходов (для master_wide)

for n, path in KV_VOPR_FILES.items():
    prefix = f"d004_vopr{n}"

    df = pd.read_csv(path, low_memory=False)
    df = add_year_household(df)

    # агрегируем расходы ДО переименования колонок (нужны настоящие имена STOIMK/PRICE)
    money_col = next((c for c in MONEY_COLS if c in df.columns), None)
    if money_col is not None:
        part = (
            df.groupby(["HOUSEHOLD_ID", "YEAR", "KVARTAL"], dropna=False)[money_col]
            .sum(min_count=1)
            .reset_index()
            .rename(columns={money_col: "TOTAL_EXPENDITURE"})
        )
        for year, g in part.groupby("YEAR"):
            agg_by_year.setdefault(year, []).append(g)
        del part

    df = prefix_columns(df, prefix)
    df["SOURCE_TABLE"] = path.stem

    # сразу дописываем каждый год в свой файл, не держим всё в памяти
    for year, g in df.groupby("YEAR"):
        out_path = OUT_DIR / f"kv_vopr_detail_{year}.csv"
        first = year not in detail_files_started
        g.to_csv(out_path, mode="w" if first else "a", header=first, index=False)
        detail_files_started.add(year)

    print(f"  обработан {path.name} (префикс {prefix}__): {df.shape}")
    del df

for year in sorted(detail_files_started):
    print(f"kv_vopr_detail_{year}.csv записан")

# ------------------------- 4. СБОРКА И СОХРАНЕНИЕ ПО ГОДАМ -------------------------
years = sorted(person_wide["YEAR"].unique())
quarters = [1, 2, 3, 4]

for year in years:
    # -- 4a. агрегированные расходы дом-ва/квартала за год (сумма по всем 12 таблицам) --
    if year in agg_by_year:
        expenditure_year = (
            pd.concat(agg_by_year[year], ignore_index=True)
            .groupby(["HOUSEHOLD_ID", "YEAR", "KVARTAL"], dropna=False)["TOTAL_EXPENDITURE"]
            .sum(min_count=1)
            .reset_index()
        )
    else:
        expenditure_year = pd.DataFrame(columns=["HOUSEHOLD_ID", "YEAR", "KVARTAL", "TOTAL_EXPENDITURE"])

    # -- 4c. master_wide_{year}.csv: человек x квартал + d006 + расходы --
    p_year = person_wide[person_wide["YEAR"] == year]
    d006_year = d006[d006["YEAR"] == year]

    p_year_q = p_year.loc[p_year.index.repeat(4)].reset_index(drop=True)
    p_year_q["KVARTAL"] = quarters * len(p_year)

    master_year = (
        p_year_q
        .merge(d006_year.drop(columns=["TE", "K"], errors="ignore"),
               on=["HOUSEHOLD_ID", "YEAR"], how="left")
        .merge(expenditure_year, on=["HOUSEHOLD_ID", "YEAR", "KVARTAL"], how="left")
    )
    master_year.to_csv(OUT_DIR / f"master_wide_{year}.csv", index=False)
    print(f"master_wide_{year}: {master_year.shape}")
    del p_year_q, master_year

print("Готово. Файлы сохранены в", OUT_DIR)

person_wide (d008+d002): (169059, 263)
d006: (46743, 249)
  обработан kv_vopr_0_combined.csv (префикс d004_vopr0__): (192000, 9)
  обработан kv_vopr_1_combined.csv (префикс d004_vopr1__): (4391820, 13)
  обработан kv_vopr_2_combined.csv (префикс d004_vopr2__): (998776, 12)
  обработан kv_vopr_3_combined.csv (префикс d004_vopr3__): (422130, 11)
  обработан kv_vopr_4_combined.csv (префикс d004_vopr4__): (37119, 11)
  обработан kv_vopr_5_combined.csv (префикс d004_vopr5__): (74202, 11)
  обработан kv_vopr_6_combined.csv (префикс d004_vopr6__): (675523, 13)
  обработан kv_vopr_7_combined.csv (префикс d004_vopr7__): (248753, 11)
  обработан kv_vopr_9_combined.csv (префикс d004_vopr9__): (917692, 23)
  обработан kv_vopr_10_combined.csv (префикс d004_vopr10__): (166793, 15)
  обработан kv_vopr_11_combined.csv (префикс d004_vopr11__): (396510, 36)
  обработан kv_vopr_12_combined.csv (префикс d004_vopr12__): (91166, 15)
kv_vopr_detail_2021.csv записан
kv_vopr_detail_2022.csv записан
kv_vopr_det

In [ ]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 8.9 MB/s eta 0:00:00


In [ ]:
all_merged_2021 = pd.read_csv("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/D00X/master_all_merged_2021.csv")

In [ ]:
all_merged_2022 = pd.read_csv("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/D00X/master_all_merged_2022.csv")

In [ ]:
all_merged_2023 = pd.read_csv("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/D00X/master_all_merged_2023.csv")

/tmp/ipykernel_654/1277854226.py:1: DtypeWarning: Columns (492) have mixed types. Specify dtype option on import or set low_memory=False.
  all_merged_2023 = pd.read_csv("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/D00X/master_all_merged_2023.csv")


In [ ]:
all_merged_2024 = pd.read_csv("/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/D00X/master_all_merged_2024.csv")

In [ ]:
all_merged_2021.isna().sum()[all_merged_2021.isna().sum() > 0]

,0
SEM_POL,50028
UROV,3176
VR_OTS,166852
PR_IZA,165116
PR_IZI,163476
...,...
TDP45,165876
TDP46,166912
ZT_d006,4552
NOMER_d006,4552


In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 2.6 MB/s eta 0:00:00


In [ ]:
df_2021 = pd.read_csv('/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/D00X/master_wide_2021.csv')

In [ ]:
df_2021.to_excel('D00X_2021.xlsx')

In [ ]:
df_2022 = pd.read_csv('/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/D00X/master_wide_2022.csv')
df_2022.to_excel('D00X_2022.xlsx')

In [ ]:
df_2023 = pd.read_csv('/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/D00X/master_wide_2023.csv')
df_2023.to_excel('D00X_2023.xlsx')

In [ ]:
df_2024 = pd.read_csv('/content/drive/MyDrive/Datathon2026/synthetic_microdata_2021-2024_20260921/D00X/master_wide_2024.csv')
df_2024.to_excel('D00X_2024.xlsx')